# Test Methylation deepTools Prototype

This notebook is a self-contained prototype for building sample-level methylation bigWigs, preparing per-tool called-region BEDs, running `computeMatrix` / `plotProfile` / `plotHeatmap`, saving manifest tables, and previewing the outputs.


## 1. Environment and imports

In [1]:
%load_ext autoreload
%autoreload 2

import os
import shutil
import sys
import tempfile
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "repo_paths.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate repo root from the current working directory.")
    PROJECT_ROOT = PROJECT_ROOT.parent

CHROMATIN_ANALYSIS_DIR = PROJECT_ROOT / "analysis" / "03_chromatin_analysis"
REGION_CALLING_UTILS_DIR = PROJECT_ROOT / "analysis" / "01_region_calling_analysis" / "utils"
matplotlib_cache_dir = Path(tempfile.gettempdir()) / f"matplotlib-{os.getuid()}"
matplotlib_cache_dir.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(matplotlib_cache_dir)

for import_path in [PROJECT_ROOT, CHROMATIN_ANALYSIS_DIR, REGION_CALLING_UTILS_DIR]:
    import_str = str(import_path)
    if import_str not in sys.path:
        sys.path.insert(0, import_str)

from repo_paths import REGION_CALLING_RESULTS_DIR
from methyl_tool_comparator import SharedPrepManager
from chromatin_analysis_utils import (
    clean_interval_df,
    filter_regions_for_deeptools,
    get_deeptools_processor_count,
    get_eligible_chrom_sizes,
    run_command,
    sample_to_sample_id,
    write_bed,
)
from figures.utils.figures_utils import (
    REGION_CALLING_TOOL_LABELS,
    REGION_CALLING_TOOL_ORDER,
    TOOL_REGISTRY as FIGURE_TOOL_REGISTRY,
    _canonical_tool_name,
    _load_bed_gz_sample,
    _load_beta_sample,
    _load_chrom_sizes,
    _write_bigwig,
    load_tool_regions,
    normalize_tool_slug,
    region_type_for_tool,
)

CANONICAL_CHROMOSOMES = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]
FIGURE_TOOL_REGISTRY_BY_NAME = {
    tool_config["tool"]: tool_config for tool_config in FIGURE_TOOL_REGISTRY
}
TOOL_PLATFORM_BY_NAME = {
    tool_name: tool_config["platform"] for tool_name, tool_config in FIGURE_TOOL_REGISTRY_BY_NAME.items()
}
ARRAY_SIGNAL_TOOLS = {"methylseg_hm450k", "dnmtools_array"}


## 2. User knobs

In [2]:
CONFIGS_PATH = PROJECT_ROOT / "analysis" / "01_region_calling_analysis" / "slurm_code" / "configs.txt"
OUT_DIR = PROJECT_ROOT / "analysis" / "01_region_calling_analysis" / "out" / "methylation_deeptools_test"

SELECTED_SAMPLES = None
SELECTED_TOOLS = list(REGION_CALLING_TOOL_ORDER)

INCLUDE_HEATMAPS = True
WGBS_BIN_SIZE = 10_000
HM450K_BIN_SIZE = 500_000
REGION_BODY_LENGTH = 1_000_000
WGBS_FLANK_LENGTH = 500_000
HM450K_FLANK_LENGTH = 2_000_000

FORCE_REBUILD_BIGWIGS = True
FORCE_RERUN_DEEPTOOLS = True

PREVIEW_SAMPLE = None

BIGWIG_DIR = OUT_DIR / "bigwigs"
PREPARED_REGION_DIR = OUT_DIR / "prepared_regions"
DEEPTOOLS_DIR = OUT_DIR / "deeptools"
SHARED_PREP_DIR = OUT_DIR / "shared_prep"
TABLES_DIR = OUT_DIR / "tables"
LOGS_DIR = OUT_DIR / "logs"
DEEPTOOLS_EXPECTED_ENV_BIN = Path(
    "/uufs/chpc.utah.edu/common/home/clementm-group1/conda/mambaforge/env/jt_wgbs_analysis/bin"
)

for directory in [OUT_DIR, BIGWIG_DIR, PREPARED_REGION_DIR, DEEPTOOLS_DIR, SHARED_PREP_DIR, TABLES_DIR, LOGS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

display(Markdown(f"Using output root: `{OUT_DIR}`"))


Using output root: `/uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/01_region_calling_analysis/out/methylation_deeptools_test`

## 3. Sample/config discovery helpers

In [3]:
def ensure_deeptools_commands() -> dict[str, str]:
    commands = {
        "computeMatrix": shutil.which("computeMatrix"),
        "plotProfile": shutil.which("plotProfile"),
        "plotHeatmap": shutil.which("plotHeatmap"),
    }
    missing = [name for name, resolved in commands.items() if resolved is None]
    if missing:
        expected_msg = ""
        if DEEPTOOLS_EXPECTED_ENV_BIN.exists():
            expected_msg = (
                f" Expected them on PATH from jt_wgbs_analysis, for example under {DEEPTOOLS_EXPECTED_ENV_BIN}."
            )
        raise RuntimeError(
            "Missing deepTools command(s): " + ", ".join(missing) + "."
            + " Start the notebook from the jt_wgbs_analysis environment."
            + expected_msg
        )
    return commands


def validate_selected_tools(selected_tools: list[str]) -> list[str]:
    available_tools = set(REGION_CALLING_TOOL_ORDER)
    deduped_tools = list(dict.fromkeys(selected_tools))
    invalid = [tool for tool in deduped_tools if tool not in available_tools]
    if invalid:
        raise ValueError(
            "Unsupported selected tools: " + ", ".join(invalid)
            + ". Choose from: " + ", ".join(REGION_CALLING_TOOL_ORDER)
        )
    if not deduped_tools:
        raise ValueError("SELECTED_TOOLS must contain at least one tool.")
    return deduped_tools


def read_config_paths(configs_path: Path) -> list[Path]:
    configs_path = Path(configs_path).expanduser().resolve()
    if not configs_path.exists():
        raise FileNotFoundError(f"Config manifest not found: {configs_path}")
    config_paths = []
    for raw_line in configs_path.read_text().splitlines():
        line = raw_line.strip()
        if not line:
            continue
        config_path = Path(line).expanduser().resolve()
        if not config_path.exists():
            raise FileNotFoundError(f"Config file listed in {configs_path} does not exist: {config_path}")
        config_paths.append(config_path)
    if not config_paths:
        raise ValueError(f"No config paths were found in {configs_path}")
    return config_paths


def load_sample_configs(configs_path: Path, selected_samples=None) -> pd.DataFrame:
    selected_sample_set = None if selected_samples is None else {str(sample) for sample in selected_samples}
    rows = []
    for config_order, config_path in enumerate(read_config_paths(configs_path)):
        with open(config_path) as handle:
            config = yaml.safe_load(handle) or {}
        sample = str(config.get("sample", "")).strip()
        meth_file = str(config.get("meth_file", "")).strip()
        genome = str(config.get("genome", "")).strip()
        if not sample or not meth_file or not genome:
            raise ValueError(
                f"Config is missing one of sample/meth_file/genome: {config_path}"
            )
        if selected_sample_set is not None and sample not in selected_sample_set:
            continue
        rows.append(
            {
                "config_order": config_order,
                "config_path": str(config_path),
                "sample": sample,
                "sample_id": sample_to_sample_id(sample),
                "meth_file": str(Path(meth_file).expanduser().resolve()),
                "genome": genome,
            }
        )
    if not rows:
        raise ValueError("No sample configs matched the current selection.")
    return pd.DataFrame(rows).sort_values("config_order").reset_index(drop=True)


def ensure_wgbstools_if_needed(sample_configs_df: pd.DataFrame) -> None:
    requires_wgbstools = sample_configs_df["meth_file"].astype(str).str.endswith(".beta").any()
    if requires_wgbstools and shutil.which("wgbstools") is None:
        raise RuntimeError(
            "Selected samples include .beta methylation files, but `wgbstools` was not found on PATH."
        )


def required_signal_tracks(selected_tools: list[str]) -> list[str]:
    signal_tracks = {"wgbs"}
    if any(tool in ARRAY_SIGNAL_TOOLS for tool in selected_tools):
        signal_tracks.add("hm450k")
    return [track for track in ["wgbs", "hm450k"] if track in signal_tracks]


def signal_track_for_tool(tool: str) -> str:
    return "hm450k" if tool in ARRAY_SIGNAL_TOOLS else "wgbs"


def bin_size_for_signal_track(signal_track: str) -> int:
    if signal_track == "hm450k":
        return int(HM450K_BIN_SIZE)
    if signal_track == "wgbs":
        return int(WGBS_BIN_SIZE)
    raise ValueError(f"Unsupported signal track: {signal_track}")


def bin_size_for_tool(tool: str) -> int:
    return bin_size_for_signal_track(signal_track_for_tool(tool))


def flank_length_for_signal_track(signal_track: str) -> int:
    if signal_track == "hm450k":
        return int(HM450K_FLANK_LENGTH)
    if signal_track == "wgbs":
        return int(WGBS_FLANK_LENGTH)
    raise ValueError(f"Unsupported signal track: {signal_track}")


def flank_length_for_tool(tool: str) -> int:
    return flank_length_for_signal_track(signal_track_for_tool(tool))


def load_beta_track_from_beta_table(beta_path: Path) -> pd.DataFrame:
    beta_path = Path(beta_path).expanduser().resolve()
    if not beta_path.exists():
        raise FileNotFoundError(f"Missing beta track file: {beta_path}")

    with open(beta_path) as fh:
        header_fields = fh.readline().rstrip("\n").split("\t")

    has_header = header_fields[:4] == ["chrom", "start", "end", "beta"]
    if has_header:
        beta_df = pd.read_csv(beta_path, sep="\t")
    else:
        n_cols = len(header_fields)
        if n_cols < 4:
            raise ValueError(f"Unsupported beta track with fewer than 4 columns: {beta_path}")
        column_names = ["chrom", "start", "end", "beta"] + [f"extra_{idx}" for idx in range(n_cols - 4)]
        beta_df = pd.read_csv(beta_path, sep="\t", header=None, names=column_names)

    beta_df = beta_df.loc[:, ["chrom", "start", "end", "beta"]].copy()
    beta_df["chrom"] = beta_df["chrom"].astype(str)
    beta_df["chrom"] = "chr" + beta_df["chrom"].str.replace("^chr", "", regex=True)
    beta_df["start"] = pd.to_numeric(beta_df["start"], errors="coerce")
    beta_df["end"] = pd.to_numeric(beta_df["end"], errors="coerce")
    beta_df["beta"] = pd.to_numeric(beta_df["beta"], errors="coerce")
    beta_df = beta_df.dropna(subset=["chrom", "start", "end", "beta"]).copy()
    beta_df = beta_df.loc[beta_df["chrom"].isin(CANONICAL_CHROMOSOMES)].copy()
    beta_df = beta_df.loc[beta_df["beta"].between(0.0, 1.0)].copy()
    beta_df["start"] = beta_df["start"].astype(int)
    beta_df["end"] = beta_df["end"].astype(int)
    beta_df = beta_df.loc[beta_df["end"] > beta_df["start"]].reset_index(drop=True)
    return beta_df


def build_shared_prep_outputs(config_row: dict):
    manager = SharedPrepManager(
        sample_id=config_row["sample"],
        meth_file=config_row["meth_file"],
        genome=config_row["genome"],
        out_dir=SHARED_PREP_DIR,
        force_recreate=FORCE_REBUILD_BIGWIGS,
        print_logs=True,
        skip_450k=False,
    )
    return manager.prepare()


def resolve_source_region_path(sample: str, tool: str) -> Path:
    canonical_tool = _canonical_tool_name(tool)
    tool_config = FIGURE_TOOL_REGISTRY_BY_NAME[canonical_tool]
    source_path = REGION_CALLING_RESULTS_DIR
    for part in tool_config["path_parts"]:
        source_path = source_path / part.format(sample=sample)
    return source_path


def sort_manifest_df(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    return df.sort_values(["config_order", "tool_order", "sample", "tool"]).reset_index(drop=True)


deeptools_commands = ensure_deeptools_commands()
selected_tools = validate_selected_tools(SELECTED_TOOLS)
sample_configs_df = load_sample_configs(CONFIGS_PATH, SELECTED_SAMPLES)
ensure_wgbstools_if_needed(sample_configs_df)
signal_tracks = required_signal_tracks(selected_tools)

display(sample_configs_df)
print("Selected tools:", selected_tools)
print("Signal tracks:", signal_tracks)
print("Samples:", sample_configs_df["sample"].tolist())


,config_order,config_path,sample,sample_id,meth_file,genome
0,0,/uufs/chpc.utah.edu/common/home/clementm-group...,SRR26107673,SRR26107673,/uufs/chpc.utah.edu/common/home/clementm-group...,hg19
1,1,/uufs/chpc.utah.edu/common/home/clementm-group...,WGBS_colon-primary-normal_1_meth,WGBS_colon-primary-normal_1_meth,/uufs/chpc.utah.edu/common/home/clementm-group...,hg38
2,2,/uufs/chpc.utah.edu/common/home/clementm-group...,WGBS_colon-primary-normal_2_meth,WGBS_colon-primary-normal_2_meth,/uufs/chpc.utah.edu/common/home/clementm-group...,hg38
3,3,/uufs/chpc.utah.edu/common/home/clementm-group...,WGBS_colon-primary-normal_3_meth,WGBS_colon-primary-normal_3_meth,/uufs/chpc.utah.edu/common/home/clementm-group...,hg38
4,4,/uufs/chpc.utah.edu/common/home/clementm-group...,WGBS_colon-primary-tumor_1_meth,WGBS_colon-primary-tumor_1_meth,/uufs/chpc.utah.edu/common/home/clementm-group...,hg38
5,5,/uufs/chpc.utah.edu/common/home/clementm-group...,WGBS_colon-primary-tumor_2_meth,WGBS_colon-primary-tumor_2_meth,/uufs/chpc.utah.edu/common/home/clementm-group...,hg38
6,6,/uufs/chpc.utah.edu/common/home/clementm-group...,WGBS_colon-primary-tumor_3_meth,WGBS_colon-primary-tumor_3_meth,/uufs/chpc.utah.edu/common/home/clementm-group...,hg38
7,7,/uufs/chpc.utah.edu/common/home/clementm-group...,ESO26.wgbs,ESO26,/uufs/chpc.utah.edu/common/home/clementm-group...,hg38
8,8,/uufs/chpc.utah.edu/common/home/clementm-group...,TE5.wgbs,TE5,/uufs/chpc.utah.edu/common/home/clementm-group...,hg38


Selected tools: ['methylseg_wgbs', 'methylseg_hm450k', 'methylseekr', 'dnmtools', 'dnmtools_array', 'dnmtools_pmr', 'mmseekr', 'methylasso']
Signal tracks: ['wgbs', 'hm450k']
Samples: ['SRR26107673', 'WGBS_colon-primary-normal_1_meth', 'WGBS_colon-primary-normal_2_meth', 'WGBS_colon-primary-normal_3_meth', 'WGBS_colon-primary-tumor_1_meth', 'WGBS_colon-primary-tumor_2_meth', 'WGBS_colon-primary-tumor_3_meth', 'ESO26.wgbs', 'TE5.wgbs']


## 4. Signal-track export

In [4]:
def load_beta_track_from_config(config_row: dict) -> pd.DataFrame:
    meth_file = Path(config_row["meth_file"]).expanduser().resolve()
    if not meth_file.exists():
        raise FileNotFoundError(f"Missing methylation input for {config_row['sample']}: {meth_file}")

    if meth_file.suffix == ".beta":
        raw_df = _load_beta_sample(meth_file, config_row["genome"])
    elif meth_file.suffixes[-2:] == [".bed", ".gz"]:
        raw_df = _load_bed_gz_sample(meth_file)
    else:
        raise ValueError(
            f"Unsupported methylation input format for {config_row['sample']}: {meth_file}"
        )

    beta_df = raw_df.rename(
        columns={
            "CpG_chrm": "chrom",
            "CpG_start": "start",
            "CpG_end": "end",
        }
    ).copy()
    beta_df = beta_df.loc[pd.to_numeric(beta_df["coverage"], errors="coerce") > 0].copy()
    beta_df["beta"] = (
        pd.to_numeric(beta_df["methylated_reads"], errors="coerce")
        / pd.to_numeric(beta_df["coverage"], errors="coerce")
    )
    beta_df = beta_df.dropna(subset=["chrom", "start", "end", "beta"]).copy()
    beta_df = beta_df.loc[:, ["chrom", "start", "end", "beta"]].reset_index(drop=True)
    return beta_df


def bin_beta_track_for_deeptools(beta_df: pd.DataFrame, genome: str, *, bin_bp: int) -> pd.DataFrame:
    if beta_df.empty:
        return beta_df.iloc[0:0].copy()

    chrom_size_map = dict(_load_chrom_sizes(genome))
    working_df = beta_df.copy()
    working_df = working_df.loc[working_df["chrom"].isin(chrom_size_map)].copy()
    if working_df.empty:
        return pd.DataFrame(columns=["chrom", "start", "end", "beta"])

    working_df["bin_start"] = (working_df["start"] // int(bin_bp)) * int(bin_bp)
    binned_df = (
        working_df.groupby(["chrom", "bin_start"], as_index=False, sort=True, observed=True)
        .agg(beta=("beta", "mean"))
        .reset_index(drop=True)
    )
    binned_df["start"] = binned_df["bin_start"].astype(int)
    binned_df["end"] = binned_df.apply(
        lambda row: min(int(row["start"]) + int(bin_bp), int(chrom_size_map[str(row["chrom"])])),
        axis=1,
    )
    binned_df = binned_df.loc[binned_df["end"] > binned_df["start"]].copy()
    return binned_df.loc[:, ["chrom", "start", "end", "beta"]].reset_index(drop=True)


def export_bigwig_for_sample_track(config_row: dict, signal_track: str, shared_prep_outputs=None) -> dict:
    sample = config_row["sample"]
    signal_bin_bp = bin_size_for_signal_track(signal_track)
    output_path = BIGWIG_DIR / f"{sample}.{signal_track}.bin{int(signal_bin_bp)}bp.methylation.bigwig"
    cache_used = output_path.exists() and not FORCE_REBUILD_BIGWIGS
    if not cache_used:
        if signal_track == "wgbs":
            beta_df = load_beta_track_from_config(config_row)
            signal_source = config_row["meth_file"]
        elif signal_track == "hm450k":
            if shared_prep_outputs is None or shared_prep_outputs.hm450k_beta is None:
                raise FileNotFoundError(
                    f"HM450K shared prep beta track is missing for {sample}."
                )
            beta_df = load_beta_track_from_beta_table(shared_prep_outputs.hm450k_beta)
            signal_source = str(shared_prep_outputs.hm450k_beta)
        else:
            raise ValueError(f"Unsupported signal track: {signal_track}")
        binned_beta_df = bin_beta_track_for_deeptools(
            beta_df,
            config_row["genome"],
            bin_bp=signal_bin_bp,
        )
        chrom_sizes = _load_chrom_sizes(config_row["genome"])
        _write_bigwig(binned_beta_df, output_path, chrom_sizes)
    else:
        signal_source = config_row["meth_file"] if signal_track == "wgbs" else (
            str(shared_prep_outputs.hm450k_beta)
            if shared_prep_outputs is not None and shared_prep_outputs.hm450k_beta is not None
            else ""
        )

    return {
        "config_order": int(config_row["config_order"]),
        "sample": sample,
        "sample_id": config_row["sample_id"],
        "genome": config_row["genome"],
        "signal_track": signal_track,
        "signal_source": signal_source,
        "signal_bin_bp": int(signal_bin_bp),
        "bigwig_path": str(output_path),
        "cache_used": cache_used,
    }


signal_rows = []
sample_failures = []

for config_row in sample_configs_df.to_dict("records"):
    shared_prep_outputs = None
    if "hm450k" in signal_tracks:
        try:
            shared_prep_outputs = build_shared_prep_outputs(config_row)
        except Exception as exc:
            sample_failures.append(
                {
                    "sample": config_row["sample"],
                    "tool": "<sample-level>",
                    "stage": "shared_prep",
                    "error": str(exc),
                }
            )
            continue

    for signal_track in signal_tracks:
        try:
            signal_rows.append(
                export_bigwig_for_sample_track(
                    config_row,
                    signal_track,
                    shared_prep_outputs=shared_prep_outputs,
                )
            )
        except Exception as exc:
            sample_failures.append(
                {
                    "sample": config_row["sample"],
                    "tool": f"<signal:{signal_track}>",
                    "stage": "signal_track_export",
                    "error": str(exc),
                }
            )

signal_manifest_df = pd.DataFrame(signal_rows).sort_values(["config_order", "sample", "signal_track"]).reset_index(drop=True)
display(signal_manifest_df)


[2026-08-14 18:44:00] Building shared prep artifacts in /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/01_region_calling_analysis/out/methylation_deeptools_test/shared_prep/SRR26107673/shared_prep
[2026-08-14 18:44:21] Command wgbstools view --genome hg19 /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/data/methylation_data/SRR26107673.beta -o /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/01_region_calling_analysis/out/methylation_deeptools_test/shared_prep/SRR26107673/shared_prep/.wgbs.tsv.tmp.2204534.1786754640569077842 | Output: /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/01_region_calling_analysis/out/methylation_deeptools_test/shared_prep/SRR26107673/logs/job_logs/wgbstools_2007477.stdout | Error: /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/01_region_calling_analysis/out/methylation_deeptools_test

,config_order,sample,sample_id,genome,signal_track,signal_source,signal_bin_bp,bigwig_path,cache_used
0,0,SRR26107673,SRR26107673,hg19,hm450k,/uufs/chpc.utah.edu/common/home/clementm-group...,500000,/uufs/chpc.utah.edu/common/home/clementm-group...,False
1,0,SRR26107673,SRR26107673,hg19,wgbs,/uufs/chpc.utah.edu/common/home/clementm-group...,10000,/uufs/chpc.utah.edu/common/home/clementm-group...,False
2,1,WGBS_colon-primary-normal_1_meth,WGBS_colon-primary-normal_1_meth,hg38,hm450k,/uufs/chpc.utah.edu/common/home/clementm-group...,500000,/uufs/chpc.utah.edu/common/home/clementm-group...,False
3,1,WGBS_colon-primary-normal_1_meth,WGBS_colon-primary-normal_1_meth,hg38,wgbs,/uufs/chpc.utah.edu/common/home/clementm-group...,10000,/uufs/chpc.utah.edu/common/home/clementm-group...,False
4,2,WGBS_colon-primary-normal_2_meth,WGBS_colon-primary-normal_2_meth,hg38,hm450k,/uufs/chpc.utah.edu/common/home/clementm-group...,500000,/uufs/chpc.utah.edu/common/home/clementm-group...,False
5,2,WGBS_colon-primary-normal_2_meth,WGBS_colon-primary-normal_2_meth,hg38,wgbs,/uufs/chpc.utah.edu/common/home/clementm-group...,10000,/uufs/chpc.utah.edu/common/home/clementm-group...,False
6,3,WGBS_colon-primary-normal_3_meth,WGBS_colon-primary-normal_3_meth,hg38,hm450k,/uufs/chpc.utah.edu/common/home/clementm-group...,500000,/uufs/chpc.utah.edu/common/home/clementm-group...,False
7,3,WGBS_colon-primary-normal_3_meth,WGBS_colon-primary-normal_3_meth,hg38,wgbs,/uufs/chpc.utah.edu/common/home/clementm-group...,10000,/uufs/chpc.utah.edu/common/home/clementm-group...,False
8,4,WGBS_colon-primary-tumor_1_meth,WGBS_colon-primary-tumor_1_meth,hg38,hm450k,/uufs/chpc.utah.edu/common/home/clementm-group...,500000,/uufs/chpc.utah.edu/common/home/clementm-group...,False
9,4,WGBS_colon-primary-tumor_1_meth,WGBS_colon-primary-tumor_1_meth,hg38,wgbs,/uufs/chpc.utah.edu/common/home/clementm-group...,10000,/uufs/chpc.utah.edu/common/home/clementm-group...,False


## 5. Region preparation

In [5]:
def build_region_manifest_row(config_row: dict, tool: str, signal_row: dict) -> dict:
    source_region_path = resolve_source_region_path(config_row["sample"], tool)
    if not source_region_path.exists():
        raise FileNotFoundError(
            f"Missing region file for {config_row['sample']} {tool}: {source_region_path}"
        )

    bigwig_path = Path(signal_row["bigwig_path"])
    chrom_sizes = get_eligible_chrom_sizes(CANONICAL_CHROMOSOMES, bigwig_path)
    if not chrom_sizes:
        raise RuntimeError(f"No eligible canonical chromosomes found in {bigwig_path}")

    raw_region_df = load_tool_regions(config_row["sample"], tool)
    prepared_region_df = clean_interval_df(raw_region_df, chrom_sizes)
    if prepared_region_df.empty:
        raise AssertionError(
            f"{config_row['sample']} {tool} produced zero retained regions after cleaning."
        )

    tool_bin_bp = bin_size_for_tool(tool)
    tool_flank_bp = flank_length_for_tool(tool)

    prepared_path = PREPARED_REGION_DIR / config_row["sample"] / f"{tool}.bed"
    prepared_path.parent.mkdir(parents=True, exist_ok=True)
    write_bed(prepared_region_df, prepared_path)

    deeptools_region_df = filter_regions_for_deeptools(prepared_region_df, tool_bin_bp)
    deeptools_region_path = (
        DEEPTOOLS_DIR
        / config_row["sample"]
        / tool
        / f"{config_row['sample']}.{tool}.deeptools_regions.bed"
    )
    deeptools_region_path.parent.mkdir(parents=True, exist_ok=True)
    write_bed(deeptools_region_df, deeptools_region_path)

    return {
        "config_order": int(config_row["config_order"]),
        "tool_order": int(selected_tools.index(tool)),
        "sample": config_row["sample"],
        "sample_id": config_row["sample_id"],
        "tool": tool,
        "tool_label": REGION_CALLING_TOOL_LABELS[tool],
        "tool_platform": TOOL_PLATFORM_BY_NAME[_canonical_tool_name(tool)],
        "signal_track": signal_row["signal_track"],
        "signal_source": signal_row["signal_source"],
        "region_type": region_type_for_tool(tool),
        "source_region_path": str(source_region_path),
        "prepared_region_path": str(prepared_path),
        "deeptools_region_path": str(deeptools_region_path),
        "bigwig_path": str(bigwig_path),
        "total_regions": int(len(prepared_region_df)),
        "visualized_regions": int(len(deeptools_region_df)),
        "excluded_short_regions": int(len(prepared_region_df) - len(deeptools_region_df)),
        "min_region_length_bp": int(tool_bin_bp),
        "deeptools_bin_bp": int(tool_bin_bp),
        "flank_length": int(tool_flank_bp),
        "region_body_length": int(REGION_BODY_LENGTH),
    }


signal_lookup = {(row["sample"], row["signal_track"]): row for row in signal_rows}
region_rows = []
failure_rows = list(sample_failures)

for config_row in sample_configs_df.to_dict("records"):
    for tool in selected_tools:
        try:
            signal_track = signal_track_for_tool(tool)
            signal_row = signal_lookup.get((config_row["sample"], signal_track))
            if signal_row is None:
                raise FileNotFoundError(
                    f"Missing {signal_track} signal track for {config_row['sample']} {tool}."
                )
            region_rows.append(build_region_manifest_row(config_row, tool, signal_row))
        except Exception as exc:
            failure_rows.append(
                {
                    "sample": config_row["sample"],
                    "tool": tool,
                    "stage": "region_preparation",
                    "error": str(exc),
                }
            )

region_manifest_df = sort_manifest_df(pd.DataFrame(region_rows))
region_manifest_path = TABLES_DIR / "region_manifest.tsv"
if not region_manifest_df.empty:
    region_manifest_df.to_csv(region_manifest_path, sep="\t", index=False)

display(region_manifest_df)
print(f"Saved region manifest to: {region_manifest_path}")


,config_order,tool_order,sample,sample_id,tool,tool_label,tool_platform,signal_track,signal_source,region_type,...,prepared_region_path,deeptools_region_path,bigwig_path,total_regions,visualized_regions,excluded_short_regions,min_region_length_bp,deeptools_bin_bp,flank_length,region_body_length
0,0,0,SRR26107673,SRR26107673,methylseg_wgbs,MethylSeg WGBS,wgbs,wgbs,/uufs/chpc.utah.edu/common/home/clementm-group...,pmd,...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,276,275,1,10000,10000,500000,1000000
1,0,1,SRR26107673,SRR26107673,methylseg_hm450k,MethylSeg HM450K,hm450k,hm450k,/uufs/chpc.utah.edu/common/home/clementm-group...,pmd,...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,905,649,256,500000,500000,2000000,1000000
2,0,2,SRR26107673,SRR26107673,methylseekr,MethylSeekR,wgbs,wgbs,/uufs/chpc.utah.edu/common/home/clementm-group...,pmd,...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,12170,10014,2156,10000,10000,500000,1000000
3,0,3,SRR26107673,SRR26107673,dnmtools,DNMTools,wgbs,wgbs,/uufs/chpc.utah.edu/common/home/clementm-group...,pmd,...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,1360,1360,0,10000,10000,500000,1000000
4,0,4,SRR26107673,SRR26107673,dnmtools_array,DNMTools Array,hm450k,hm450k,/uufs/chpc.utah.edu/common/home/clementm-group...,pmd,...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,53,52,1,500000,500000,2000000,1000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66,8,3,TE5.wgbs,TE5,dnmtools,DNMTools,wgbs,wgbs,/uufs/chpc.utah.edu/common/home/clementm-group...,pmd,...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,2496,2496,0,10000,10000,500000,1000000
67,8,4,TE5.wgbs,TE5,dnmtools_array,DNMTools Array,hm450k,hm450k,/uufs/chpc.utah.edu/common/home/clementm-group...,pmd,...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,255,112,143,500000,500000,2000000,1000000
68,8,5,TE5.wgbs,TE5,dnmtools_pmr,DNMTools PMR,wgbs,wgbs,/uufs/chpc.utah.edu/common/home/clementm-group...,pmr,...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,39219,408,38811,10000,10000,500000,1000000
69,8,6,TE5.wgbs,TE5,mmseekr,MMSeekR,wgbs,wgbs,/uufs/chpc.utah.edu/common/home/clementm-group...,pmd,...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,4223,4191,32,10000,10000,500000,1000000


Saved region manifest to: /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/01_region_calling_analysis/out/methylation_deeptools_test/tables/region_manifest.tsv


## 6. deepTools execution

In [ ]:
def maybe_run_command(command, expected_outputs, force=False):
    expected_paths = [Path(path) for path in expected_outputs]
    if not force and expected_paths and all(path.exists() for path in expected_paths):
        return True
    run_command(command, expected_outputs=expected_paths)
    return False


def run_deeptools_for_region(region_row: dict) -> dict:
    if int(region_row["visualized_regions"]) <= 0:
        raise RuntimeError(
            f"{region_row['sample']} {region_row['tool']} had zero regions at least {region_row['deeptools_bin_bp']} bp after filtering."
        )

    sample = str(region_row["sample"])
    sample_id = str(region_row["sample_id"])
    tool = str(region_row["tool"])
    tool_label = str(region_row["tool_label"])
    sample_output_dir = DEEPTOOLS_DIR / sample / tool
    sample_output_dir.mkdir(parents=True, exist_ok=True)

    matrix_path = sample_output_dir / f"{sample}.{tool}.methylation.matrix.gz"
    matrix_values_path = sample_output_dir / f"{sample}.{tool}.methylation.matrix.tsv"
    sorted_regions_path = sample_output_dir / f"{sample}.{tool}.sorted_regions.bed"
    profile_path = sample_output_dir / f"{sample}.{tool}.profile.png"
    heatmap_path = sample_output_dir / f"{sample}.{tool}.heatmap.png"

    profile_title = f"{sample}: methylation across {tool_label} called regions"
    heatmap_title = f"{sample}: methylation heatmap across {tool_label} called regions"

    compute_matrix_command = [
        deeptools_commands["computeMatrix"],
        "scale-regions",
        "-p",
        str(get_deeptools_processor_count()),
        "-S",
        str(region_row["bigwig_path"]),
        "-R",
        str(region_row["deeptools_region_path"]),
        "-b",
        str(region_row["flank_length"]),
        "-a",
        str(region_row["flank_length"]),
        "--binSize",
        str(region_row["deeptools_bin_bp"]),
        "--regionBodyLength",
        str(REGION_BODY_LENGTH),
        "--missingDataAsZero",
        "--sortRegions",
        "keep",
        "-o",
        str(matrix_path),
        "--outFileSortedRegions",
        str(sorted_regions_path),
        "--outFileNameMatrix",
        str(matrix_values_path),
    ]
    compute_matrix_cache_used = maybe_run_command(
        compute_matrix_command,
        [matrix_path, matrix_values_path, sorted_regions_path],
        force=FORCE_RERUN_DEEPTOOLS,
    )

    profile_command = [
        deeptools_commands["plotProfile"],
        "--numPlotsPerRow",
        "1",
        "-m",
        str(matrix_path),
        "--perGroup",
        "--samplesLabel",
        sample_id,
        "--regionsLabel",
        tool_label,
        "--startLabel",
        "Region start",
        "--endLabel",
        "Region end",
        "--plotTitle",
        profile_title,
        "--plotWidth",
        "11",
        "--plotHeight",
        "6",
        "--dpi",
        "200",
        "-out",
        str(profile_path),
    ]
    profile_cache_used = maybe_run_command(
        profile_command,
        [profile_path],
        force=FORCE_RERUN_DEEPTOOLS,
    )

    heatmap_cache_used = None
    saved_heatmap_path = ""
    if INCLUDE_HEATMAPS:
        heatmap_command = [
            deeptools_commands["plotHeatmap"],
            "-m",
            str(matrix_path),
            "--samplesLabel",
            sample_id,
            "--regionsLabel",
            tool_label,
            "--startLabel",
            "Region start",
            "--endLabel",
            "Region end",
            "--plotTitle",
            heatmap_title,
            "--sortRegions",
            "keep",
            "--heatmapWidth",
            "12",
            "--heatmapHeight",
            "10",
            "--whatToShow",
            "heatmap and colorbar",
            "-out",
            str(heatmap_path),
        ]
        heatmap_cache_used = maybe_run_command(
            heatmap_command,
            [heatmap_path],
            force=FORCE_RERUN_DEEPTOOLS,
        )
        saved_heatmap_path = str(heatmap_path)

    return {
        **region_row,
        "matrix_path": str(matrix_path),
        "matrix_values_path": str(matrix_values_path),
        "sorted_regions_path": str(sorted_regions_path),
        "profile_path": str(profile_path),
        "heatmap_path": saved_heatmap_path,
        "profile_title": profile_title,
        "heatmap_title": heatmap_title,
        "compute_matrix_cache_used": compute_matrix_cache_used,
        "profile_cache_used": profile_cache_used,
        "heatmap_cache_used": heatmap_cache_used,
    }


deeptools_rows = []
for region_row in region_manifest_df.to_dict("records"):
    try:
        deeptools_rows.append(run_deeptools_for_region(region_row))
    except Exception as exc:
        failure_rows.append(
            {
                "sample": region_row["sample"],
                "tool": region_row["tool"],
                "stage": "deeptools_execution",
                "error": str(exc),
            }
        )

deeptools_outputs_df = sort_manifest_df(pd.DataFrame(deeptools_rows))
deeptools_outputs_path = TABLES_DIR / "deeptools_outputs.tsv"
if not deeptools_outputs_df.empty:
    deeptools_outputs_df.to_csv(deeptools_outputs_path, sep="\t", index=False)

display(
    deeptools_outputs_df[
        [
            "sample",
            "tool",
            "tool_label",
            "signal_track",
            "matrix_path",
            "profile_path",
            "heatmap_path",
            "visualized_regions",
        ]
    ]
)
print(f"Saved deepTools outputs manifest to: {deeptools_outputs_path}")


$ /uufs/chpc.utah.edu/common/home/clementm-group1/conda/mambaforge/env/jt_wgbs_analysis/bin/computeMatrix scale-regions -p 1 -S /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/01_region_calling_analysis/out/methylation_deeptools_test/bigwigs/SRR26107673.wgbs.bin10000bp.methylation.bigwig -R /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/01_region_calling_analysis/out/methylation_deeptools_test/deeptools/SRR26107673/methylseg_wgbs/SRR26107673.methylseg_wgbs.deeptools_regions.bed -b 500000 -a 500000 --binSize 10000 --regionBodyLength 1000000 --missingDataAsZero --sortRegions keep -o /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/01_region_calling_analysis/out/methylation_deeptools_test/deeptools/SRR26107673/methylseg_wgbs/SRR26107673.methylseg_wgbs.methylation.matrix.gz --outFileSortedRegions /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/

## 7. Manifest summaries and inline display

In [ ]:
failures_df = pd.DataFrame(failure_rows)

expected_combinations = len(sample_configs_df) * len(selected_tools)
successful_combinations = len(deeptools_outputs_df)
print(f"Successful sample/tool combinations: {successful_combinations} / {expected_combinations}")

display(Markdown("### Region manifest"))
display(region_manifest_df)
display(Markdown("### deepTools outputs manifest"))
display(deeptools_outputs_df)

if failures_df.empty:
    display(Markdown("### Failures\nNo sample/tool failures were recorded."))
else:
    display(Markdown("### Failures"))
    display(failures_df)

preview_candidates = deeptools_outputs_df["sample"].drop_duplicates().tolist()
preview_sample = PREVIEW_SAMPLE if PREVIEW_SAMPLE is not None else (preview_candidates[0] if preview_candidates else None)

if preview_sample is None:
    display(Markdown("### Preview\nNo successful deepTools outputs are available to preview yet."))
else:
    preview_df = deeptools_outputs_df.loc[
        deeptools_outputs_df["sample"].astype(str) == str(preview_sample)
    ].copy()
    preview_df = preview_df.sort_values(["tool_order", "tool"]).reset_index(drop=True)
    display(Markdown(f"### Preview sample: `{preview_sample}`"))
    for row in preview_df.itertuples(index=False):
        display(Markdown(f"#### {row.tool_label} ({row.region_type.upper()}, signal={row.signal_track})"))
        display(Markdown(f"Matrix: `{row.matrix_path}`"))
        display(Image(filename=row.profile_path))
        if INCLUDE_HEATMAPS and str(row.heatmap_path).strip():
            display(Image(filename=row.heatmap_path))
